# 🏗️ PyQt6 – Lekce 9: OOP přístup a QMainWindow

Dosud jsme psali kód procedurálně – vše v jednom bloku.
Pro reálné aplikace ale používáme **Objektově Orientovaný přístup (OOP)**.
Zároveň přecházíme od `QWidget` na `QMainWindow`, který přináší
standardní strukturu profesionálních desktopových aplikací.

---

## ⚙️ Procedurální vs OOP přístup

```python
# ❌ Procedurální – obtížně rozšiřitelné
app = QApplication.instance() or QApplication([])
okno = QWidget()
tlacitko = QPushButton("Klikni", okno)
okno.show()
app.exec()

# ✅ OOP – přehledné, znovupoužitelné
class MojeOkno(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Moje Aplikace")
        # ... setup widgetů
```

---

## 📦 Struktura QMainWindow

```
┌─────────────────────────────────────┐
│  QMenuBar  (menu: Soubor, Úpravy…)  │
├─────────────────────────────────────┤
│  QToolBar  (ikony nástrojů)         │
├─────────────────────────────────────┤
│                                     │
│   Central Widget  ← sem dáme obsah  │
│                                     │
├─────────────────────────────────────┤
│  QStatusBar  (stavový řádek dole)   │
└─────────────────────────────────────┘
```

| Část | Metoda pro nastavení |
|---|---|
| Central Widget | `self.setCentralWidget(widget)` |
| Menu Bar | `self.menuBar()` |
| Tool Bar | `self.addToolBar(...)` |
| Status Bar | `self.statusBar().showMessage("text")` |

---

## 🟢 Ukázka 1 – Převod procedurálního kódu na třídu

Porovnáme **stejnou aplikaci** napsanou dvěma způsoby.

In [1]:
# ✅ OOP verze – přehledná třída dědí z QWidget
from PyQt6.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel

class PocitadloApp(QWidget):
    def __init__(self):
        super().__init__()           # Zavolá konstruktor QWidget
        self.pocet = 0               # Stav aplikace – atribut instance
        self.setWindowTitle("Počítadlo (OOP)")
        self.resize(300, 100)
        self._sestav_ui()            # Pomocná metoda pro setup UI

    def _sestav_ui(self):
        layout = QVBoxLayout()
        self.setLayout(layout)

        self.napis = QLabel("Kliknuto: 0×")   # self. → přístupné v celé třídě
        self.tlacitko = QPushButton("Klikni!")
        self.tlacitko.clicked.connect(self.kliknuto)  # Metoda jako slot

        layout.addWidget(self.napis)
        layout.addWidget(self.tlacitko)

    def kliknuto(self):              # Slot = metoda třídy
        self.pocet += 1
        self.napis.setText(f"Kliknuto: {self.pocet}×")


app = QApplication.instance() or QApplication([])
okno = PocitadloApp()   # Vytvoříme instanci naší třídy
okno.show()
app.exec()

0

---

## 🟢 Ukázka 2 – QMainWindow se StatusBarem

`QMainWindow` přidá automaticky menu bar, toolbar a status bar.
Obsah okna musíme umístit do **central widget**.

In [2]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QPushButton, QLabel
)

class HlavniOkno(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("QMainWindow ukázka")
        self.resize(400, 200)

        # --- Central Widget ---
        central = QWidget()             # Kontejner pro obsah
        self.setCentralWidget(central)  # Registrace jako střed okna

        layout = QVBoxLayout()
        central.setLayout(layout)

        self.napis = QLabel("Zatím nic...")
        tlacitko = QPushButton("Klikni")
        tlacitko.clicked.connect(self.po_kliknuti)

        layout.addWidget(self.napis)
        layout.addWidget(tlacitko)

        # --- Status Bar ---
        self.statusBar().showMessage("Připraven")  # Stavový řádek dole

    def po_kliknuti(self):
        self.napis.setText("Tlačítko bylo stisknuto!")
        self.statusBar().showMessage("Akce provedena")  # Aktualizace statusu


app = QApplication.instance() or QApplication([])
okno = HlavniOkno()
okno.show()
app.exec()

0

---

## 🟡 Ukázka 3 – Dědičnost: Rozšíření existující třídy

OOP nám umožňuje **dědit** z vlastní třídy a přidávat nové funkce
bez přepisování původního kódu.

In [3]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QPushButton, QLabel, QLineEdit
)

# Základní třída s obecnou funkcionalitou
class ZakladniOkno(QMainWindow):
    def __init__(self, titulek="Okno"):
        super().__init__()
        self.setWindowTitle(titulek)
        self.resize(400, 120)
        self.central = QWidget()
        self.setCentralWidget(self.central)
        self.layout_hlavni = QVBoxLayout()
        self.central.setLayout(self.layout_hlavni)

    def pridej_widget(self, widget):
        """Pomocná metoda – přidá widget do hlavního layoutu"""
        self.layout_hlavni.addWidget(widget)

# Odvozená třída – přidáme vstupní pole
class PrihlasovaniOkno(ZakladniOkno):
    def __init__(self):
        super().__init__(titulek="Přihlášení")  # Volá ZakladniOkno.__init__
        self._sestav()

    def _sestav(self):
        self.vstup = QLineEdit()
        self.vstup.setPlaceholderText("Zadejte jméno...")
        self.vysledek = QLabel("")
        btn = QPushButton("Přihlásit se")
        btn.clicked.connect(self.prihlasit)

        self.pridej_widget(self.vstup)    # Použijeme zděděnou metodu
        self.pridej_widget(btn)
        self.pridej_widget(self.vysledek)

    def prihlasit(self):
        jmeno = self.vstup.text().strip()
        if jmeno:
            self.vysledek.setText(f"Vítej, {jmeno}! 👋")
            self.statusBar().showMessage(f"Přihlášen: {jmeno}")
        else:
            self.vysledek.setText("⚠️ Zadejte jméno!")


app = QApplication.instance() or QApplication([])
okno = PrihlasovaniOkno()
okno.show()
app.exec()

0

---

## 🔴 Ukázka 4 – Zavírání okna s potvrzením (`closeEvent`)

`QMainWindow` poskytuje speciální metody (tzv. **event handlery**),
které se zavolají při systémových událostech. Díky nim můžeme
například zobrazit dialog před zavřením okna.

In [4]:
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget, QVBoxLayout, QLabel, QMessageBox

class OknoSPotvrzenim(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Pokus zavřít ✕")
        self.resize(350, 100)
        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)
        layout.addWidget(QLabel("Zkus kliknout na ✕ v rohu okna."))

    def closeEvent(self, event):
        """Přetížení closeEvent – zavolá se vždy před zavřením okna"""
        odpoved = QMessageBox.question(
            self, "Konec",
            "Opravdu chcete ukončit aplikaci?",
            QMessageBox.StandardButton.Yes | QMessageBox.StandardButton.No
        )
        if odpoved == QMessageBox.StandardButton.Yes:
            event.accept()   # Povolíme zavření
        else:
            event.ignore()   # Zrušíme zavření


app = QApplication.instance() or QApplication([])
okno = OknoSPotvrzenim()
okno.show()
app.exec()

0

---

## 📋 Shrnutí lekce

| Koncept | Co si pamatovat |
|---|---|
| `class Okno(QMainWindow)` | Dědíme z QMainWindow/QWidget |
| `super().__init__()` | Vždy volat v `__init__`! |
| `self.widget` | Widgety ukládat jako atributy (`self.`) |
| `self.setCentralWidget(w)` | Povinné pro `QMainWindow` |
| `self.statusBar()` | Spodní stavový řádek |
| `closeEvent(event)` | Přetížení → kód před zavřením okna |

## ✏️ Úkoly

1. Napište třídu `KalkulackaApp(QMainWindow)` se dvěma vstupy (`QDoubleSpinBox`) a čtyřmi tlačítky (+, −, ×, ÷). Výsledek zobrazte v `QLabel`.
2. Přidejte do třídy `closeEvent`, který se zeptá na potvrzení před zavřením.
3. Vytvořte základní třídu `FormularOkno` se jménem a e-mailem a odvoďte z ní třídu `RegistraceOkno` s heslem navíc.